# 🎬 Netflix Recommendation Engine
### EDA, Preprocessing & Collaborative Filtering Walkthrough

---
**Author:** Adithya Anil
  
**Libraries:** Pandas, NumPy, Scikit-learn, Matplotlib, Seaborn  
**Algorithm:** User-Based Collaborative Filtering with Cosine Similarity


## 0. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Blues_r')

print('All imports successful ✓')

## 1. Load Dataset

In [ ]:
# If you don't have the dataset, run generate_sample_data.py first
# !python ../generate_sample_data.py

df = pd.read_csv('../data/movies.csv')
print(f'Shape: {df.shape}')
df.head(10)

## 2. Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Missing Values ===')
print(df.isnull().sum())
print('\n=== Basic Statistics ===')
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rating distribution
sns.histplot(df['rating'], bins=10, kde=True, ax=axes[0], color='#2E75B6')
axes[0].set_title('Rating Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Rating')
axes[0].set_ylabel('Count')

# Ratings per user
ratings_per_user = df.groupby('userId')['rating'].count()
sns.histplot(ratings_per_user, bins=20, ax=axes[1], color='#1F3864')
axes[1].set_title('Ratings per User', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Number of Ratings')
axes[1].set_ylabel('Number of Users')

plt.tight_layout()
plt.show()

In [ ]:
# Genre frequency
genre_series = df['genres'].str.split('|').explode()
genre_counts = genre_series.value_counts().head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=genre_counts.values, y=genre_counts.index, palette='Blues_r')
plt.title('Top 15 Genres by Frequency', fontsize=14, fontweight='bold')
plt.xlabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# Average rating per movie (top 15)
avg_rating = df.groupby('title')['rating'].agg(['mean','count']).reset_index()
avg_rating.columns = ['title','avg_rating','num_ratings']
top_movies = avg_rating[avg_rating['num_ratings'] >= 5].sort_values('avg_rating', ascending=False).head(15)

plt.figure(figsize=(12, 6))
sns.barplot(data=top_movies, x='avg_rating', y='title', palette='Blues_r')
plt.title('Top 15 Highest Rated Movies (min 5 ratings)', fontsize=14, fontweight='bold')
plt.xlabel('Average Rating')
plt.tight_layout()
plt.show()
top_movies

## 3. Preprocessing

In [ ]:
# Drop missing values
df_clean = df.dropna().reset_index(drop=True)
print(f'Rows before: {len(df)} | After dropping NaN: {len(df_clean)}')

In [ ]:
# Label Encoding — genres
le = LabelEncoder()
df_clean['genres_encoded'] = le.fit_transform(df_clean['genres'].astype(str))
print('Genres encoded:')
df_clean[['genres','genres_encoded']].drop_duplicates().head(10)

In [ ]:
# MinMax Scaling — ratings
scaler = MinMaxScaler()
df_clean['rating_scaled'] = scaler.fit_transform(df_clean[['rating']])
print('Rating scaling sample:')
df_clean[['rating','rating_scaled']].head(8)

## 4. Build User-Movie Matrix

In [ ]:
user_movie_matrix = df_clean.pivot_table(
    index='userId',
    columns='title',
    values='rating',
    fill_value=0
)
print(f'User-Movie Matrix: {user_movie_matrix.shape[0]} users × {user_movie_matrix.shape[1]} movies')
user_movie_matrix.head(5)

## 5. Compute Cosine Similarity

In [ ]:
similarity_matrix = cosine_similarity(user_movie_matrix)
user_similarity_df = pd.DataFrame(
    similarity_matrix,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)
print(f'Similarity matrix: {user_similarity_df.shape}')
user_similarity_df.head(5)

In [ ]:
# Heatmap of similarity (first 20 users)
plt.figure(figsize=(12, 10))
subset = user_similarity_df.iloc[:20, :20]
sns.heatmap(subset, cmap='Blues', annot=False, linewidths=0.3)
plt.title('User-User Cosine Similarity Heatmap (first 20 users)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Generate Recommendations

In [ ]:
def recommend_for_user(user_id, top_n=10, n_neighbours=20):
    """Full recommendation pipeline for a given user."""
    # Find similar users
    similar_users = (
        user_similarity_df[user_id]
        .drop(user_id)
        .sort_values(ascending=False)
        .head(n_neighbours)
    )

    # Movies already watched
    user_row     = user_movie_matrix.loc[user_id]
    already_seen = set(user_row[user_row > 0].index)

    # Score unseen movies
    scores = {}
    for neighbour, similarity in similar_users.items():
        for movie, rating in user_movie_matrix.loc[neighbour].items():
            if movie not in already_seen and rating > 0:
                scores[movie] = scores.get(movie, 0.0) + similarity * rating

    # Sort and display
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    result = pd.DataFrame(ranked, columns=['Movie', 'Score'])
    result.index += 1
    return result

# Try it out
TARGET_USER = 1
recommendations = recommend_for_user(TARGET_USER, top_n=10)
print(f'\nTop 10 Recommendations for User {TARGET_USER}:')
recommendations

## 7. Model Evaluation (RMSE & MAE)

In [ ]:
train, test = train_test_split(df_clean, test_size=0.2, random_state=42)

# Rebuild matrix and similarity on training data
train_matrix = train.pivot_table(index='userId', columns='title', values='rating', fill_value=0)
train_sim    = pd.DataFrame(
    cosine_similarity(train_matrix),
    index=train_matrix.index,
    columns=train_matrix.index
)

predictions, actuals = [], []

for _, row in test.iterrows():
    user, movie, actual = row['userId'], row['title'], row['rating']
    if user in train_sim.index and movie in train_matrix.columns:
        sim_scores    = train_sim[user]
        movie_ratings = train_matrix[movie]
        numerator     = np.dot(sim_scores, movie_ratings)
        denominator   = np.sum(np.abs(sim_scores))
        pred = numerator / denominator if denominator != 0 else 0
        predictions.append(pred)
        actuals.append(actual)

rmse = np.sqrt(mean_squared_error(actuals, predictions))
mae  = np.mean(np.abs(np.array(actuals) - np.array(predictions)))

print(f'Predictions made : {len(predictions)}')
print(f'RMSE             : {rmse:.4f}')
print(f'MAE              : {mae:.4f}')

In [ ]:
# Predicted vs Actual scatter
plt.figure(figsize=(8, 6))
plt.scatter(actuals, predictions, alpha=0.4, color='#2E75B6', s=20)
plt.plot([0, 5], [0, 5], 'r--', label='Perfect prediction')
plt.xlabel('Actual Rating')
plt.ylabel('Predicted Rating')
plt.title(f'Actual vs Predicted Ratings  |  RMSE: {rmse:.4f}', fontsize=13, fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

---
## ✅ Summary

| Step | Action | Result |
|---|---|---|
| EDA | Analysed rating distributions, genre frequencies | Identified data patterns |
| Label Encoding | Encoded `genres` column | Numeric representation |
| MinMax Scaling | Scaled `rating` to [0, 1] | Normalised features |
| User-Movie Matrix | Pivot table (users × movies) | Structured input for CF |
| Cosine Similarity | Pairwise user similarity | Neighbourhood identification |
| Recommendations | Weighted scoring of unseen movies | Personalised Top-N list |
| Evaluation | Train/test split RMSE + MAE | Model performance quantified |
